<a href="https://colab.research.google.com/github/JariusF10-8/bt4222/blob/main/notebooks/features/history_mode_check_chris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How should the training CTR features be built?

Test features must not use test clicks, so at test time the counts are always out of date. This notebook checks how the training features should be built so that a model still does well on out-of-date counts.

`feature_engineering_chris.ipynb` exports these versions with the same columns:

| File | Training | Validation | Test |
|---|---|---|---|
| `candidate_history_online` | up to the impression | up to the impression | uses earlier test clicks, not used |
| `candidate_history_nightly` | up to midnight of that day | up to midnight of that day | same as `frozen` |
| `candidate_history_frozen` | up to the impression | up to 13 Nov midnight | up to 15 Nov midnight |
| `candidate_history_mixed` | half the impressions up to date, half nightly | not built | not built |

The same LightGBM LambdaRank model is trained on the same non-history features plus one version of the history features, and each version is scored on the same validation impressions. Only validation labels are used.

Validation is scored two ways. `nightly` validation counts are at most one day old, like the one-day test set. `frozen` validation counts are up to two days old, so it is a harder check.

Run after `feature_engineering_chris.ipynb`. It takes about 20 minutes and 5 GB RAM on a Colab CPU.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules


def find_project_root():
    configured = os.environ.get('BT4222_PROJECT_ROOT')
    options = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents]
    for folder in options:
        if (folder / 'src/data_loading.py').is_file():
            return folder.resolve()
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None and IN_COLAB:
    import io
    import tempfile
    import zipfile
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
    target = Path(tempfile.mkdtemp(prefix='bt4222_', dir='/content'))
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(target)
    PROJECT_ROOT = next(target.rglob('src/data_loading.py')).parent.parent
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(PROJECT_ROOT / 'requirements.txt')], check=True)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook inside the repository or set BT4222_PROJECT_ROOT.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb
print('Repository found:', PROJECT_ROOT.name, '| LightGBM', lgb.__version__)

Repository found: bt4222-main | LightGBM 4.7.0


In [2]:
import gc
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from src.history_features import FEATURE_COLUMNS as HISTORY_COLUMNS
from src.feature_diagnostics import impression_metrics, paired_difference

warnings.filterwarnings('ignore', message='X does not have valid feature names')
pd.set_option('display.width', 200)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SHARED = Path('/content/drive/MyDrive/BT4222 Group Project')
    zipeng_v2 = SHARED / 'feature_engineering_zp/data/processed/v2'
    default_processed = zipeng_v2 if (zipeng_v2 / 'manifest.json').is_file() else SHARED / 'feature_engineering_chris/data/processed/v1'
    os.environ.setdefault('BT4222_PROCESSED_DIR', str(default_processed))
    os.environ.setdefault('BT4222_CHRIS_FEATURES_DIR', str(SHARED / 'feature_engineering_chris/outputs/features/chris_v1'))

PROCESSED_DIR = Path(os.environ.get('BT4222_PROCESSED_DIR', PROJECT_ROOT / 'data/processed/v1'))
FEATURES_DIR = Path(os.environ.get('BT4222_CHRIS_FEATURES_DIR', PROJECT_ROOT / 'outputs/features/chris_v1'))
needed = [FEATURES_DIR / 'train/candidate_history_mixed.csv.gz', FEATURES_DIR / 'validation/candidate_history_nightly.csv.gz']
if not all(path.is_file() for path in needed):
    raise FileNotFoundError('Run feature_engineering_chris.ipynb first so the history files exist.')
started = time.time()

## 1. Load features

Each candidate-level file is checked row by row against `candidates.csv`, so every feature lines up with the right impression and candidate.

In [3]:
STATIC_COLUMNS = ['candidate_history_subcategory_share', 'candidate_history_subcategory_count', 'history_subcategory_missing',
                  'category_id', 'subcategory_id', 'has_entity_at_title_start', 'lead_group',
                  'title_entity_first_position', 'abstract_entity_first_position', 'title_entity_absent', 'abstract_entity_absent']
CATEGORICAL = ['category_id', 'subcategory_id', 'lead_group']
LEAD_GROUPS = {'none': 0, 'person': 1, 'organisation': 2, 'place': 3, 'other': 4}


def read_keyed(partition, name, columns):
    frame = pd.read_csv(FEATURES_DIR / partition / f'{name}.csv.gz', dtype={'event_key': 'string', 'candidate_position': 'int32'})
    keys = data[partition]['keys']
    if not (frame['event_key'].to_numpy() == keys['event_key'].to_numpy()).all() or \
       not (frame['candidate_position'].to_numpy() == keys['candidate_position'].to_numpy()).all():
        raise ValueError(f'{partition}/{name} is not aligned with candidates.csv')
    return frame[columns].to_numpy(dtype='float32')


data = {}
for partition in ('train', 'validation'):
    candidates = pd.read_csv(PROCESSED_DIR / partition / 'candidates.csv', keep_default_na=False,
                             dtype={'event_key': 'string', 'news_id': 'string', 'candidate_position': 'int32', 'label': 'int8'})
    data[partition] = {'keys': candidates[['event_key', 'candidate_position']], 'label': candidates['label'].to_numpy()}
    match = read_keyed(partition, 'candidate_subcategory_match', STATIC_COLUMNS[:3])
    cats = pd.read_csv(FEATURES_DIR / partition / 'news_category_encoding.csv', dtype={'news_id': 'string'})
    prom = pd.read_csv(FEATURES_DIR / partition / 'news_entity_prominence.csv', dtype={'news_id': 'string'}, keep_default_na=False)
    prom['lead_group'] = prom['title_start_type_group'].map(LEAD_GROUPS)
    article = cats.merge(prom, on='news_id', validate='one_to_one')
    joined = candidates[['news_id']].merge(article, on='news_id', how='left', validate='many_to_one')
    data[partition]['static'] = np.hstack([match, joined[STATIC_COLUMNS[3:]].to_numpy(dtype='float32')])
    events = candidates['event_key'].to_numpy()
    change = np.flatnonzero(events[1:] != events[:-1]) + 1
    data[partition]['groups'] = np.diff(np.concatenate([[0], change, [len(events)]]))
    if len(data[partition]['groups']) != candidates['event_key'].nunique():
        raise ValueError('Candidates of an impression must be contiguous')
    data[partition]['events'] = events

versions = {'train': ['online', 'nightly', 'mixed'], 'validation': ['online', 'nightly', 'frozen']}
for partition, names in versions.items():
    for name in names:
        data[partition][name] = read_keyed(partition, f'candidate_history_{name}', HISTORY_COLUMNS)
    gc.collect()
print(f"Train rows {len(data['train']['label']):,} in {len(data['train']['groups']):,} impressions; "
      f"validation rows {len(data['validation']['label']):,} in {len(data['validation']['groups']):,} impressions")
print(f'Loaded in {time.time() - started:.0f}s')

Train rows 3,389,342 in 95,071 impressions; validation rows 2,454,102 in 61,894 impressions
Loaded in 44s


## 2. Train each version

Every version uses the same settings: 300 trees, learning rate 0.05, 63 leaves and a fixed seed. The settings are not tuned. Only the history features change.

In [4]:
PARAMS = dict(objective='lambdarank', n_estimators=300, learning_rate=0.05, num_leaves=63, min_child_samples=200,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)


def features(partition, version):
    static = data[partition]['static']
    return static if version is None else np.hstack([data[partition][version], static])


def fit(version):
    names = ([] if version is None else HISTORY_COLUMNS) + STATIC_COLUMNS
    model = lgb.LGBMRanker(**PARAMS)
    model.fit(features('train', version), data['train']['label'], group=data['train']['groups'],
              feature_name=names, categorical_feature=CATEGORICAL)
    return model


VARIANTS = [
    ('no history features', None, [None]),
    ('fresh training', 'online', ['nightly', 'frozen', 'online']),
    ('nightly training', 'nightly', ['nightly', 'frozen']),
    ('mixed training', 'mixed', ['nightly', 'frozen']),
]
scores, rows, models = {}, [], {}
for label, train_version, val_versions in VARIANTS:
    tick = time.time()
    model = fit(train_version)
    models[label] = model
    for val_version in val_versions:
        name = f'{label} | val {val_version or "no history"}'
        scores[name] = model.predict(features('validation', val_version))
        frame = pd.DataFrame({'event_key': data['validation']['events'], 'label': data['validation']['label'], 's': scores[name]})
        rows.append({'training': label, 'validation features': val_version or '-', **impression_metrics(frame, 's')})
    print(f'{label}: trained and scored in {time.time() - tick:.0f}s')
results = pd.DataFrame(rows)
display(results)

no history features: trained and scored in 147s


fresh training: trained and scored in 248s


nightly training: trained and scored in 204s


mixed training: trained and scored in 203s


,training,validation features,impressions,AUC,MRR,nDCG@5,nDCG@10
0,no history features,-,61894,0.6226,0.2932,0.3118,0.3710
1,fresh training,nightly,61894,0.6235,0.2921,0.3115,0.3711
2,fresh training,frozen,61894,0.6117,0.2870,0.3056,0.3642
3,fresh training,online,61894,0.7002,0.3443,0.3748,0.4318
4,nightly training,nightly,61894,0.6357,0.2977,0.3181,0.3773
5,nightly training,frozen,61894,0.6258,0.2940,0.3134,0.3728
6,mixed training,nightly,61894,0.6369,0.2993,0.3202,0.3800
7,mixed training,frozen,61894,0.6289,0.2964,0.3168,0.3757


## 3. Paired comparisons

Each row compares two versions on the same validation impressions: the mean per-impression difference (second minus first) with a 95% interval. If the interval does not include 0, the gap is bigger than the noise.

In [5]:
scored = pd.DataFrame({'event_key': data['validation']['events'], 'label': data['validation']['label'], **scores})
pairs = [
    ('no history features | val no history', 'fresh training | val frozen'),
    ('fresh training | val frozen', 'nightly training | val frozen'),
    ('fresh training | val frozen', 'mixed training | val frozen'),
    ('fresh training | val nightly', 'nightly training | val nightly'),
    ('fresh training | val nightly', 'mixed training | val nightly'),
    ('nightly training | val nightly', 'mixed training | val nightly'),
    ('no history features | val no history', 'mixed training | val nightly'),
]
paired = pd.DataFrame([{'first': a, 'second': b, **paired_difference(scored, a, b, metric)}
                       for a, b in pairs for metric in ('AUC', 'nDCG@10')])
display(paired)
importance = pd.Series(models['mixed training'].booster_.feature_importance('gain'),
                       index=HISTORY_COLUMNS + STATIC_COLUMNS).sort_values(ascending=False)
display((importance / importance.sum()).round(3).head(12).rename('share of gain (mixed training)'))
results.to_csv(FEATURES_DIR / 'diagnostic_history_mode_check.csv', index=False)
paired.to_csv(FEATURES_DIR / 'diagnostic_history_mode_paired.csv', index=False)
print(f'Total {(time.time() - started) / 60:.1f} minutes')

,first,second,metric,mean_difference,ci_low,ci_high,impressions
0,no history features | val no history,fresh training | val frozen,AUC,-0.0109,-0.0132,-0.0086,61894
1,no history features | val no history,fresh training | val frozen,nDCG@10,-0.0067,-0.0086,-0.0049,61894
2,fresh training | val frozen,nightly training | val frozen,AUC,0.0141,0.0120,0.0161,61894
3,fresh training | val frozen,nightly training | val frozen,nDCG@10,0.0085,0.0068,0.0103,61894
4,fresh training | val frozen,mixed training | val frozen,AUC,0.0172,0.0153,0.0191,61894
5,fresh training | val frozen,mixed training | val frozen,nDCG@10,0.0115,0.0098,0.0131,61894
6,fresh training | val nightly,nightly training | val nightly,AUC,0.0121,0.0102,0.0141,61894
7,fresh training | val nightly,nightly training | val nightly,nDCG@10,0.0062,0.0045,0.0079,61894
8,fresh training | val nightly,mixed training | val nightly,AUC,0.0134,0.0115,0.0153,61894
9,fresh training | val nightly,mixed training | val nightly,nDCG@10,0.0089,0.0073,0.0105,61894


subcategory_id                         0.245
art_prior_ctr                          0.122
art_ctr_24h                            0.117
candidate_history_subcategory_share    0.105
candidate_history_subcategory_count    0.046
art_hours_since_first_exposure         0.043
user_article_prior_exposures           0.040
abstract_entity_first_position         0.031
subcat_prior_ctr                       0.027
art_exposures_1h                       0.026
title_entity_first_position            0.022
lead_group                             0.021
Name: share of gain (mixed training), dtype: float64

Total 16.3 minutes


## 4. Results

| Training file | Validation `nightly` AUC / nDCG@10 | Validation `frozen` AUC / nDCG@10 |
|---|---|---|
| no history features | 0.6226 / 0.3710 | 0.6226 / 0.3710 |
| `online` (up to date) | 0.6235 / 0.3711 | 0.6117 / 0.3642 |
| `nightly` | 0.6357 / 0.3773 | 0.6258 / 0.3728 |
| `mixed` (half up to date, half nightly) | **0.6369 / 0.3800** | **0.6289 / 0.3757** |

With up-to-date validation features the up-to-date model reaches AUC 0.7002 and nDCG@10 0.4318.

1. **Training on up-to-date counts fails on out-of-date counts.** It is worse than leaving the history features out (AUC -0.0109 [-0.0132, -0.0086], nDCG@10 -0.0067 [-0.0086, -0.0049]). The model learns that counts are always up to date, then meets articles with no counts.
2. **Matching how out of date the counts are fixes it.** Nightly training beats up-to-date training by +0.0121 AUC and +0.0062 nDCG@10 on nightly validation, and by +0.0141 and +0.0085 on frozen validation.
3. **Mixing is best.** Mixed training beats nightly training by +0.0013 AUC [0.0001, 0.0024] and +0.0028 nDCG@10 [0.0016, 0.0039], and beats up-to-date training by +0.0172 AUC and +0.0115 nDCG@10 on frozen validation. Seeing both kinds of counts probably teaches the model how much to trust a count.
4. **History features are worth keeping.** With mixed training they add +0.0143 AUC and +0.0091 nDCG@10 over no history features. The top features by gain are `subcategory_id`, `art_prior_ctr`, `art_ctr_24h` and `candidate_history_subcategory_share`.
5. **Real-time logging would help a lot**, about +0.05 nDCG@10. This is for the report discussion only.

**What this means for the group.** The shared feature bundle uses `train_only` features: training rows are up to date, and validation and test rows only use training-period clicks. That is the `online` training row scored on `frozen` validation above, which did worse than no history features. `mixed` training only uses training clicks, so it follows the same rule and is the fix to use if a model includes these features. In the candidate attention model, adding the `train_only` CTR features gave no validation gain (nDCG@10 0.3939 against 0.3942 without them), which agrees with this check. The current models leave these features out, and `mixed` training is future work.

**Limits.** One model type with fixed settings and one mixing seed. `mixed` training was not tested inside a neural model.